# VII) Topology optimization - slot model

In [ ]:
# import packages in JupyterLite
import ngsolve
import matplotlib
import scipy

## 1) Geometry

First, we define the geometry, material distribution, and generate the mesh.

In [ ]:
pole_pairs = 6
bundles_per_half_slot = 5

# Generate the corresponding mesh
from utils.geometry import machine_mesh
mesh = machine_mesh(p=pole_pairs, 
                    bundles_per_half_slot=bundles_per_half_slot, 
                    hBundle=0.25e-3
                    )

print(f"Generated mesh with {mesh.nv} nodes, {mesh.ne} elements")

________
## 2) Background problem

See notebook [6_slot_model](6_slot_model.ipynb).

### 2.a) Material properties

In [ ]:
mur_iron = 1000           # Relative magnetic permeability of iron core
sigma_copper = 5.8e7      # Copper conductivity (S/m)
Br = 1                    # Remanent flux density of magnets (T)


# Define space-varying material coefficients
from ngsolve import pi
mu0 = 4e-7 * pi
nu = mesh.MaterialCF({"core_stator" : 1 / (mu0 * mur_iron)}, default = 1/mu0)     # reluctivity
sigma = mesh.MaterialCF({"slot(.*)_bundle.*" : sigma_copper})                     # conductivity
from utils.physics import magnetization_halbach
Mcplx = mesh.MaterialCF({"rotor" : magnetization_halbach(br = Br, p=pole_pairs)}) # magnetization

### 2.b) Current supply

In [ ]:
freq = 1000 # Electrical frequency (Hz)
Jrms = 10e6 # Current density (A/m²)
phi = 150   # load angle (°), chosen to maximize average torque
winding_type = "distributed" # "distributed" or "concentrated"

from ngsolve import Integrate
from utils.supply import phase_current, winding_arrangement, bundle_arrangement

S_bundle = Integrate(1, mesh.Materials("slot11_bundle0"))
Irms = Jrms * S_bundle
phase = phase_current(I_rms=Irms,  load_angle=phi*pi/180)
winding = winding_arrangement(phase, type = winding_type)
bundles_background = bundle_arrangement(winding = winding, 
                             bundles_per_half_slot = bundles_per_half_slot,
                             background = True)

### 2.c) Finite element space

In [ ]:
curve_order = 1   # order = 1 leads to fast simulations, order = 2 leads to more accurate results
fem_order = curve_order

from ngsolve import H1, Periodic
fes_ref = Periodic( H1(mesh.Curve(curve_order), 
                   order = fem_order, 
                   dirichlet =  "shaft|out", 
                   complex = True),  [-1]*7 )
print(f"Number of degrees of freedom of the reference FE space: {fes_ref.ndof}")

### 2.d) Solve background

In [ ]:
from utils.physics import solve_magnetoharmonic

result_background = solve_magnetoharmonic(
    fes = fes_ref, 
    frequency = 0, # we don't want any perturbation in the conductors
    reluctivity = nu,
    magnetization = Mcplx,
    conductivity=1,
    supply = bundles_background,
    verbose = 1)

a_background = result_background["solution"]["a"]

______
## 3) Slot model

### 3.a) Reduced computational region

In [ ]:
slot = "slot2"

slot_domain = slot + ".*"           # all slot
slot_air_domain = slot + ".{1}"     # slot without conductor (to save assembly time)

from utils.geometry import mask
mask_slot = mask(mesh, slot_domain)
mask_slot_air = mask(mesh, slot_air_domain)

### 3.b) Trace extraction

We use then $h_t = n\times h \in H^1(\Omega)$ (we assume stronger regularity for consistency, cf Dular paper)

$$ \int_{\partial \Omega} a^* h_t = \int_{\Omega} \text{Curl}(a^*) \cdot \nu \text{Curl}(a) $$

The interesting dofs are only the ones on the boundary, so we can assemble only on a single layer and extract a small sub-matrix.

Proceeding this way the projection is consistent (why? to be checked. write the subproblem weak formulation).

In [ ]:
slot_bnd = slot + "1.*|" + slot + "2.*"

fes_ht = H1(mesh.Curve(curve_order), 
                   order = fem_order, 
                   definedon = slot_air_domain,
                   dirichlet = slot_bnd,
                   complex = True)

from utils.physics import dual_trace
ht_background = dual_trace(fes_ht, slot_bnd, nu, a_background)


### 3.c) Reduced slot problem

Now that we have our backgroung field and its trace on the boundaries of the slot, we can simulate and obtain the same result in the slot only.

We have the freedom to chose Dirichlet or Neumann boundary conditions; or more generally

$$ \alpha a + (1-\alpha) \nu \text{Curl}(a) \times n = \alpha a_d + (1-\alpha) h_t $$

with $\alpha \in[0,1]$.

In [ ]:
choice = "full_neumann" # "full_dirichlet" or "full_neumann" or "mixed"

fes_slot = H1(mesh.Curve(curve_order), 
                   order = fem_order, 
                   definedon = slot_domain,
                   complex = True)

eps_dirichlet = 1e-10
airgap_bnd  = slot + ".*shoe"
iron_bnd  = slot + ".*lateral|" +slot + ".*bottom"

if choice == "full_dirichlet":
    alpha = mesh.BoundaryCF({iron_bnd : 1-eps_dirichlet , airgap_bnd : 1-eps_dirichlet})
elif choice == "full_neumann":
    alpha = mesh.BoundaryCF({iron_bnd : 0 , airgap_bnd : 0})
elif choice == "mixed":
    alpha = mesh.BoundaryCF({iron_bnd : 0 , airgap_bnd : 1-eps_dirichlet})

bundles_ref_slot = bundle_arrangement(winding = winding, 
                                      bundles_per_half_slot = bundles_per_half_slot,
                                      only_in= slot_domain)


_________
## 4) Point-wise optimization of the conductivity in the slot only

Too high conductivity leads to high AC losses, while too low conductivity leads to high DC losses. There is an optimum point that can be found. An optimization of the whole conductivity of a single material coil was conducted in [3_single_material_optimization](3_single_material_optimization.ipynb), and the optimization of the conductivity of each bundle independantly was performed in [4_multi_material_optimization](4_multi_material_optimization.ipynb).

However, additive manufacturing may enable even more freedom, so that the conductivity can take any value in the whole cross section.

To accelerate the computation we reduce the computation domain to a single slot.

### 3.a) Setup of the optimization problem

In [ ]:
# Problem parameters
sigma0 = sigma_copper         # initial conductivity
sigma_min = sigma_copper/10   # minimum admissible conductivity
sigma_max = sigma_copper      # maximum admissible conductivity

# Conductivity space
from ngsolve import L2, GridFunction
fes_sigma = fes_sigma = L2(mesh, order = 0, definedon = slot + ".*_bundle.*")
x0 = GridFunction(fes_sigma)
x0.Set(sigma0)

In [ ]:
# Definition of the functions of interest

from utils.physics import solve_magnetoharmonic, joule_losses
def state_function(conductivity):
    """ Returns the solution of the magnetoharmonic problem for a given conductivity value """
    result = solve_magnetoharmonic(fes = fes_slot,             # localized inside the slot
                                   frequency = freq,
                                   reluctivity = nu,     
                                   magnetization = Mcplx,
                                   conductivity = conductivity,
                                   supply = bundles_ref_slot,  # true supply
                                   # slot boundary conditions
                                   robin_bnd = slot_bnd,
                                   robin_coeff = alpha,
                                   a_dirichlet = a_background,
                                   h_tangential = ht_background,
                                   fix1dof=True,
                                   verbose = 0)
    return result

def objective_function(result):
    """ Total Joule losses to minimize """
    return joule_losses(result)

In [ ]:
# Algorithm parameters

from copy import copy
from numpy import sign

def descent(grad):
    """ Extract descent direction from the gradient """
    descent = copy(grad)
    descent.vec.data = - 1e7 * sign(grad.vec)
    return descent

# Derivative
from utils.optimization import d_joule_losses
from ngsolve import InnerProduct, CoefficientFunction as CF

def grad_joule_losses_local(result):
    """ Compute the gradient of the objective function w.r.t the conductivity value at each point """
    fes_sigma = result["info"]["conductivity"].space
    dx_global = GridFunction(fes_sigma)
    dx_global.Set(1)
    df = d_joule_losses(result, dx_global)
    grad_f = GridFunction(fes_sigma)
    grad_f.vec.data = df.vec
    return grad_f

# Derivative
def grad_joule_losses_bundle(result):
    """ Compute the gradient of the objective function w.r.t the conductivity value of each bundle """
    fes_sigma = result["info"]["conductivity"].space
    dx_global = GridFunction(fes_sigma)
    dx_global.Set(1)
    df = d_joule_losses(result, dx_global)
    # we want now to provide a conductivity field representing the steepest admissible ascent direction (we call this the "gradient")
    dfdx = CF(0)
    for bundle in result["info"]["supply"].keys():
        dx = GridFunction(fes_sigma)
        dx.Set(mesh.MaterialCF({bundle : 1})) # dx is a unit perturbation of sigma in a single bundle
        dfdx += InnerProduct(df.vec, dx.vec) * copy(dx)  
    grad_f = GridFunction(fes_sigma)
    grad_f.Set(dfdx.Compile())
    return grad_f

### 3.b) Optimization loop

In [ ]:
from utils.optimization import gradient_descent

results_topopt_sigma = gradient_descent(state=state_function,
                                 objective = objective_function,
                                 d_objective = grad_joule_losses_local,
                                 x0 = x0,
                                 x_min = sigma_min,
                                 x_max = sigma_max,
                                 descent = descent)

### 3.c) Results

In [ ]:
# Display the optimal conductivity distribution
from numpy import nan
from ngsolve.webgui import Draw
result_optim_global = results_topopt_sigma["solution"][-1] * mask_slot
Draw(result_optim_global, results_topopt_sigma["solution"][-1].space.mesh,
     settings = {"Objects" : {"Wireframe" : False}, "Colormap" : {"ncolors" : 32}},
     min = sigma_min, max = sigma_max,
     )

In [ ]:
# Display convergence
import matplotlib.pyplot as plt
print(f"=> Joule losses = {results_topopt_sigma['objective'][-1]:.2e} W/m")
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.plot(results_topopt_sigma["objective"], 'b-')
ax2.semilogy(results_topopt_sigma["criterion"], 'r-')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Joule losses (W/m)', color='b')
ax2.set_ylabel('Stop criterion', color='r')
plt.title(f"Convergence, Pj = {results_topopt_sigma['objective'][-1]:.2e} W/m")
plt.show()